In [2]:
import pandas as pd 
import requests
import json
import orjson
import pickle
import re
import torch
import numpy as np
from transformers import BartTokenizer, BartForConditionalGeneration
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [3]:
print("PyTorch version:", torch.__version__)
print('CUDA available: ', torch.cuda.is_available())
print('Num GPUs Available:', torch.cuda.device_count())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

PyTorch version: 2.6.0+cu126
CUDA available:  True
Num GPUs Available: 1
cuda


### Load User Reviews

In [ ]:
def get_review_dataframe(config):
    # reviews = read_jsonl_fast(f"data/{config['CATEGORY']}.jsonl")
    # data = {
    #     "user_id": [review['user_id'] for review in reviews],
    #     "asin": [review['asin'] for review in reviews],
    #     "review_title": [review['title'] for review in reviews],
    #     "review_text": [review['text'] for review in reviews],
    #     "rating": [review['rating'] for review in reviews],
    #     "verified_purchase": [review['verified_purchase'] for review in reviews]
    # }
    # df = pd.DataFrame(data)
    # df.to_pickle(f"data/{config['CATEGORY']}.pkl")
    df_review = pd.read_pickle(f"data/{config['CATEGORY']}.pkl")
    return df_review

In [22]:
def has_letter(text):
    return bool(re.search(r'[a-zA-Z]', text))

# 清理資料(未被驗證購買、重複資料、評論內容無英文字者)
def clean_review(df):
    df = df[df['verified_purchase']==True]  # 刪除未被驗證購買的評論
    df = df.drop_duplicates() # 刪除重複的資料
    df = df[df['review_text'].apply(has_letter)] # 去除所有不包含任何英文字的評論
    print ('去除掉所有不包含任何英文字的評論')

    return df

In [ ]:
# 迭代過濾item跟user(依評論數量門檻)
def iter_filter_review_num(df, ITEM_THRESHOLD, USER_THRESHOLD, MAX_COUNT):
    """
    篩選每個item擁有的評論/user發表的評論，不得低於N則。
    方法一(2分鐘):
    reviews.groupby(['user_id']).filter(lambda x : len(x)>3)

    方法二(14秒):
    reviews[reviews.groupby('user_id').user_id.transform('count')>3]
 
    方法三(8秒)(最快=>用這個)
    counts = reviews['user_id'].value_counts()
    reviews[reviews['user_id'].isin(counts[counts>3].index)]
    """
    C = 0
    item_min_counts, user_min_counts = 0, 0
    while (C<MAX_COUNT) and ((item_min_counts<ITEM_THRESHOLD) or (user_min_counts<USER_THRESHOLD)):
        item_counts = df['asin'].value_counts()
        df = df[df['asin'].isin(item_counts[item_counts>=ITEM_THRESHOLD].index)]  # 布林索引

        user_counts = df['user_id'].value_counts()
        df = df[df['user_id'].isin(user_counts[user_counts>=USER_THRESHOLD].index)]

        item_min_counts = df['asin'].value_counts().min()
        user_min_counts = df['user_id'].value_counts().min()
        C+=1
    print ('總迭代次數:',C)
    print ('最後每個ITEM最少有幾筆:', item_min_counts)
    print ('最後每個USER最少有幾筆:', user_min_counts)

    # # check the number of reviews for each user/item 
    # print(df['user_id'].value_counts())
    # print(item_reviews = df['asin'].value_counts())

    # # 排序結果
    # sorted_df = df.sort_values(by='user_id')
    # print(sorted_df)

    return df

In [24]:
def show_review_summary(df):
    print ("共{}筆, {}個Item, {}個User, Like比例{}".format(df.shape[0], len(df.asin.unique()), len(df.user_id.unique()), round(df.like.mean(),3)))

# 去除掉全都是不喜歡的反社會user&喜歡比例太高的user
def filter_like_ratio(df, config):

    df["like"] = df["rating"] >= 4
    show_review_summary(df)
    
    LIKE_LOWER_THRESHOLD = config['LIKE_LOWER_THRESHOLD']
    user_like_mean = df.groupby('user_id')['like'].mean()
    i = user_like_mean[user_like_mean<LIKE_LOWER_THRESHOLD].index
    num = df[df.user_id.isin(i)].shape[0]
    df = df[~df.user_id.isin(i)] 
    print ("共{}個User的評論喜歡比例小於{}，拿掉這些後df_review少了{}筆".format(len(i), LIKE_LOWER_THRESHOLD, num))

    LIKE_UPPER_THRESHOLD = config['LIKE_UPPER_THRESHOLD']
    user_like_mean = df.groupby('user_id')['like'].mean()
    i = user_like_mean[user_like_mean>LIKE_UPPER_THRESHOLD].index
    num = df[df.user_id.isin(i)].shape[0]
    df = df[~df.user_id.isin(i)] 
    print ("共{}個User的評論喜歡比例超過{}，拿掉這些後df_review少了{}筆".format(len(i), LIKE_UPPER_THRESHOLD, num))
    show_review_summary(df)
    
    return df

In [25]:
 # 定義句子拆分函數
def clean_and_split_review(review, min_words_sent=3):
    sentences = review.splitlines()  # 先以換行切句
    sentences = list(filter(None, sentences))  # 去空行
    all_sents = []
    for sent in sentences:
        sent = re.split(r' *[\.\?!][\'"\)\]]* *', sent)
        all_sents.extend(sent)
    # 過濾過短的句子
    clean_sents = [sent for sent in all_sents if len(sent.split()) > min_words_sent]
    return clean_sents

In [ ]:
def filter_reviews(df, config):  # game已先刪 但要換成items再加進來(item資料集)

    print ('原本有{}個item, {}個user, {}筆評論'.format(len(df.asin.unique()), len(df.user_id.unique().tolist()), df.shape[0]))

    # 清理資料(未被驗證購買、重複資料、評論內容無英文字者)
    df = clean_review(df)
    
    # 去除掉全都是不喜歡的反社會user&喜歡比例太高的user
    df = filter_like_ratio(df, config)

    # 迭代過濾item跟user(依評論數量門檻)
    ITEM_THRESHOLD = config['ITEM_THRESHOLD'] # 限定每個item至少要有N則評論
    USER_THRESHOLD = config['USER_THRESHOLD'] # 限定每個user至少要有N個評論
    print ('限制每個遊戲至少要有{}個評論, 每個user至少要有{}個評論 ：'.format(ITEM_THRESHOLD, USER_THRESHOLD))
    df = iter_filter_review_num(df, ITEM_THRESHOLD, USER_THRESHOLD, MAX_COUNT=1)

    # 將Review拆成以句為單位 & 過濾過短句子 => 新欄位split_review_text
    df['split_review_text'] = df['review_text'].apply(clean_and_split_review)
    # 去除split_review_text欄位為空者
    empty_list_rows = df['split_review_text'].apply(lambda x: len(x) == 0) 
    df = df[~empty_list_rows]
    print ('拆成以句子為單位並過濾過短句子(小於3個字元)&去除過濾後為空者，評論剩:',df.shape[0])

    # 再度迭代過濾item跟user(依評論數量門檻)
    ITEM_THRESHOLD = config['ITEM_THRESHOLD'] # 限定每個item至少要有N則評論
    USER_THRESHOLD = config['USER_THRESHOLD'] # 限定每個user至少要有N個評論
    print ('限制每個遊戲至少要有{}個評論, 每個user至少要有{}個評論 ：'.format(ITEM_THRESHOLD, USER_THRESHOLD))
    df = iter_filter_review_num(df,ITEM_THRESHOLD, USER_THRESHOLD, MAX_COUNT=10)

    # 再度去除掉全都是不喜歡的反社會user&喜歡比例太高的user
    df = filter_like_ratio(df, config)
    
    df.reset_index(drop=True,inplace=True)
    df.to_pickle(f"data/Clothing_Shoes_and_Jewelry_after_filter.pkl") 
    print ('儲存過濾後的dataframe.')

    user_counts = df['user_id'].value_counts()
    item_counts = df['asin'].value_counts()
    filteredID = {'user_id':user_counts[user_counts>=USER_THRESHOLD].index.tolist(), 'asin':item_counts[item_counts>=ITEM_THRESHOLD].index.tolist()}
    # filteredID.to_pickle(f"data/filteredID.pkl")  # 錯 ∵ filteredID是一個 dict(Python字典本身並沒有.to_pickle()這個方法，這是pandas的DataFrame專用方法)
    with open("data/filteredID.pkl", "wb") as f: # 對
        pickle.dump(filteredID, f)
    print ('儲存過濾後的user/item ID.')

    return df

#### Review Dataframe

In [27]:
from xmlrpc.client import Boolean

def build_and_save_review(config):
    # 根據評論數過濾
    if 'test' in config['REVIEW_FILENAME']: # 拿筆數少的資料來測試
        df = pd.read_pickle('data/reviews_test.pkl')
    else:
        df = get_review_dataframe(config)
        df = filter_reviews(df, config)
        df = df.astype({'asin':str, 'user_id':str})
        df.to_pickle('data/reviews_{}_{}.pkl'.format(config['ITEM_THRESHOLD'], config['USER_THRESHOLD']))

#### Split Train Test


In [ ]:
def generate_negative_samples(df, user_col='user_id', item_col='asin', like_col='like', n_neg_per_user=100, random_state=42):
    """
    為每個 user 產生指定數量的負樣本（like=False），並與原始資料合併回傳。

    參數：
    - df: 原始互動資料，必須包含 user_id, asin, like 欄位
    - n_neg_per_user: 每個 user 要抽幾個沒互動過的 item 當負樣本
    """
    np.random.seed(random_state)

    all_users = df[user_col].unique()
    all_items = df[item_col].unique()
    
    neg_samples = []

    for user in all_users:
        user_items = set(df[df[user_col] == user][item_col])
        candidate_items = list(set(all_items) - user_items)

        if len(candidate_items) < n_neg_per_user:
            sampled_items = candidate_items  # 全拿
        else:
            sampled_items = np.random.choice(candidate_items, n_neg_per_user, replace=False)
        
        # 驗證：不能有交集
        assert np.isin(sampled_items, list(user_items)).sum() == 0, \
            f"錯誤! User {user} 的負樣本和實際互動過的 item 有重疊！"

        for item in sampled_items:
            neg_samples.append({
                user_col: user,
                item_col: item,
                like_col: False
            })

    neg_df = pd.DataFrame(neg_samples)

    # 合併回原始資料（正樣本）
    df_with_neg = pd.concat([df, neg_df], ignore_index=True)
    return df_with_neg


In [ ]:
def split_train_val_test(config):    
    """
    先把正讚評論挑出來，切30%給test。
    因為test裡面不需要放負評資料
    (在評估test時，每個使用者會隨機補到N個遊戲當作dislike去算TOPK)
    剩下的(70%正讚&所有倒讚)再分成train, val
    """
    np.random.seed(config["RANDOM_SEED"])
    df_review = pd.read_pickle('data/reviews_{}_{}.pkl'.format(config['ITEM_THRESHOLD'], config['USER_THRESHOLD']))

    train_idx, val_idx, test_idx = [], [], []
    print ('splitting train test...')

    for u in df_review.user_id.unique():
        try:
            user_data = df_review[df_review.user_id==u]
            # 只選出like==True的資料（正樣本）然後對這一群做train/test split => 最後test裡全是like==True的樣本
            _, test = train_test_split(user_data[user_data.like==True], test_size=0.3, random_state=config['RANDOM_SEED'])
            rest = pd.concat([_, user_data[user_data.like==False]], axis=0)
            train, val = train_test_split(rest, stratify = rest['like'], test_size=0.1, random_state=config['RANDOM_SEED'])

            user_train_idx, user_val_idx, user_test_idx = list(train.index), list(val.index), list(test.index)
            if val.like.sum()==0: # 若因為正讚評論很少，導致val全都是倒讚，那就跟train換一筆
                one_like_in_train_idx = np.random.choice(list(train[train.like==True].index), 1)[0]
                one_dislike_in_val_idx = np.random.choice(list(val.index), 1)[0]
                user_train_idx.remove(one_like_in_train_idx)
                user_train_idx.append(one_dislike_in_val_idx)
                user_val_idx.remove(one_dislike_in_val_idx)
                user_val_idx.append(one_like_in_train_idx)
            train_idx += user_train_idx
            val_idx += user_val_idx
            test_idx += user_test_idx
            print (train.like.mean(), val.like.mean(), test.like.mean())


            # # 對user_data中的所有資料做抽樣 並依照like欄位的比例進行stratified（分層）抽樣，確保 True/False 的比例維持原樣
            # train, val_and_test = train_test_split(user_data, stratify = user_data['like'], test_size=0.4, random_state=5566)
            # val, test = train_test_split(val_and_test, stratify = val_and_test['like'], test_size=0.75, random_state=5566)
            # user_train_idx, user_val_idx, user_test_idx = list(train.index), list(val.index), list(test.index)
            
            # # 把test中不喜歡的評論都搬到train，搬幾筆就補幾筆train/val中喜歡的評論進來
            # # 避免把train的某些user喜歡評論掏空了= =所以改成從train/val平均抽
            # dislike_in_test = list(test[test.like==False].index)  # test中所有不喜歡的評論
            
            # # Step 4: 抽正樣本補回 test（從 train/val 抽）
            # like_train_candidates = list(train[train.like == True].index)
            # like_val_candidates = list(val[val.like == True].index)
            # # 保險：每邊抽一半
            # half = len(dislike_in_test) // 2
            # like_in_train = list(np.random.choice(like_train_candidates, min(half, len(like_train_candidates)), replace=False))
            # like_in_val = list(np.random.choice(like_val_candidates, len(dislike_in_test) - len(like_in_train), replace=False))

            # user_test_idx = [x for x in user_test_idx if x not in dislike_in_test]   # 將test中不喜歡的評論去除
            # user_train_idx = [x for x in user_train_idx if x not in like_in_train]
            # user_val_idx = [x for x in user_val_idx if x not in like_in_val]

            # # 開搬
            # user_train_idx += dislike_in_test
            # user_test_idx += like_in_train
            # user_test_idx += like_in_val
            
            # train_idx += user_train_idx
            # val_idx += user_val_idx
            # test_idx += user_test_idx
            # # print (train.like.mean(), val.like.mean(), test.like.mean())

        except Exception as e:
            print (e)
            print (u)
            print (train.shape[0], train.like.sum())
            print (val.shape[0], val.like.sum())
            print (test.shape[0], test.like.sum())
            break

    train_df, val_df, test_df = df_review.loc[train_idx], df_review.loc[val_idx], df_review.loc[test_idx]
    print ('Like ratio:', round(train_df.like.mean(), 4), round(val_df.like.mean(), 4), round(test_df.like.mean(), 4))
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    # test的每個使用者補到150個遊戲。需1分鐘
    N = config['N_ITEMS_FOR_TOPK_RECOMMENDATION']  # 150 (也可以試100)
    all_item_list = train_df.asin.unique()
    if  N > 0:
        print ('Filling up to {} items for every user in testing set...'.format(N))
        for u in test_df.user_id.unique():
            item_in_this_user = test_df[test_df.user_id==u].asin
            n = N - len(item_in_this_user) # 要補幾筆他明明沒買過item給他，並標記為不喜歡
            item_not_in_this_user = all_item_list[~np.isin(all_item_list, item_in_this_user)] # 從所有item中去掉該使用者有買過的
            n_item = np.random.choice(item_not_in_this_user, n, replace=False)

            assert np.isin(item_in_this_user, n_item).sum()==0, "錯誤!填補並標記為不喜歡的item與該使用者曾購買的item有重疊!"  # np.isin(item_in_this_user, n_item)會回傳一個布林陣列，表item_in_this_user裡的每個item是否出現在n_item中 .sum()會回傳有幾個True，也就是「重複的 item 數量」
            test_df = pd.concat([test_df, pd.DataFrame({'user_id':u, 'asin':n_item, 'like':False})], ignore_index=True)
        print ('Like ratio:', round(train_df.like.mean(), 4), round(val_df.like.mean(), 4), round(test_df.like.mean(), 4))

    train_df.to_pickle('data/ready_for_model/{}/train_df.pkl'.format(config['REVIEW_FILENAME']))
    val_df.to_pickle('data/ready_for_model/{}/val_df.pkl'.format(config['REVIEW_FILENAME']))
    test_df.to_pickle('data/ready_for_model/{}/test_df.pkl'.format(config['REVIEW_FILENAME']))
    print ('Finish splitting.')



### LDA

In [14]:
from transformers import BertTokenizer, BertModel, BertConfig

configuration = BertConfig()
bert_model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states = True).to(device)
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', use_fast=True)
args = {
        "device" : device,
        # "data_dir" : r'../data/filtered_reviews_with_split.pkl',
        # "data_chunks_dir" : r'../data/chunks',
        "emb_dim" : 768,
        "max_word" : 100,
        "max_sentence" : 10,
        "max_group": 5, # exclude default group
        "max_review_user" : 50,
        "max_review_item" : 50,
        "epoch" : 5,
        "batch_size": 32,
        "bert_configuration" : configuration,
        "bert_model" : bert_model,
        "bert_tokenizer" : bert_tokenizer
    }

In [15]:
from nltk.stem.porter import PorterStemmer
from gensim import corpora, models
from sklearn.feature_extraction.text import TfidfVectorizer

# # Modify LDA part from 書呈
# def stemmer_with_delete_stopword(split_sentences):
#     vectorizer = TfidfVectorizer(stop_words = "english")
#     stop_list = list(vectorizer.get_stop_words())
#     stop_list.extend(["im", "didnt", "thi", "doe", "did"])
#     print(stop_list)
#     porter_stemmer = PorterStemmer()
#     all_stem_sents=[]
#     for review in split_sentences:
#         review_stem_list = []
#         for sent in review:
#             sent_stem_list =[]
#             for word in sent.split(" "):
#                 word = re.sub(r'[^\w\s]', '', word)
#                 stemmed_word = porter_stemmer.stem(word.lower())
#                 if len(word) > 2 and (word not in stop_list) and (stemmed_word not in stop_list):
#                     sent_stem_list.append(stemmed_word)
#             review_stem_list.append(sent_stem_list)
#         all_stem_sents.append(review_stem_list) 
#     return all_stem_sents

# from nltk.stem.snowball import SnowballStemmer
# def stemmer_with_delete_stopword(split_sentences):
#     vectorizer = TfidfVectorizer(stop_words="english")
#     stop_list = list(vectorizer.get_stop_words())
#     stop_list.extend(["im", "didnt", "thi", "doe", "did"])

#     stemmer = SnowballStemmer("english")  # 改用 SnowballStemmer
#     all_stem_sents = []
#     for review in split_sentences:
#         review_stem_list = []
#         for sent in review:
#             sent_stem_list = []
#             for word in sent.split(" "):
#                 word = re.sub(r'[^\w\s]', '', word)
#                 stemmed_word = stemmer.stem(word.lower())  # Snowball Stemming
#                 if len(word) > 2 and (word not in stop_list) and (stemmed_word not in stop_list):
#                     sent_stem_list.append(stemmed_word)
#             review_stem_list.append(sent_stem_list)
#         all_stem_sents.append(review_stem_list)

#     return all_stem_sents

import nltk
nltk.download('wordnet')
nltk.download('stopwords')
from nltk.stem import WordNetLemmatizer
def lemmatizer_with_delete_stopword(split_sentences):
    vectorizer = TfidfVectorizer(stop_words="english")
    stop_list = list(vectorizer.get_stop_words())
    stop_list.extend(["im", "didnt", "thi", "doe", "did"])

    lemmatizer = WordNetLemmatizer()
    all_lemma_sents = []
    for review in split_sentences:
        review_lemma_list = []
        for sent in review:
            sent_lemma_list = []
            for word in sent.split(" "):
                word = re.sub(r'[^\w\s]', '', word)
                lemma_word = lemmatizer.lemmatize(word.lower())  # 進行 Lemmatization
                if len(word) > 2 and (word not in stop_list) and (lemma_word not in stop_list):
                    sent_lemma_list.append(lemma_word)
            review_lemma_list.append(sent_lemma_list)
        all_lemma_sents.append(review_lemma_list)
    
    return all_lemma_sents


# def LDAGrouping(reviews, groups):
#     all_sents = []
#     for review in reviews:
#         for sentence in review:
#             all_sents.append(sentence)
#     dictionary = corpora.Dictionary(all_sents)
#     corpus = [dictionary.doc2bow(sent) for sent in all_sents]
#     lda_model = models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=groups)
#     group_results = []
#     for sents in reviews:
#         single_corpus = [dictionary.doc2bow(sent) for sent in sents]
#         sents_group_result = []
#         for scores in lda_model.inference(single_corpus)[0]:
#             # scores.argmax()+1 --> Retain group:0 for no meaning sentences
#             sents_group_result.append(scores.argmax()+1)
#         group_results.append(sents_group_result)
#     return group_results, lda_model

def LDAGrouping(reviews, groups):
    all_sents = [sentence for review in reviews for sentence in review]
    dictionary = corpora.Dictionary(all_sents)
    corpus = [dictionary.doc2bow(sent) for sent in all_sents]

    # 訓練 LDA 模型，使用更穩定的超參數
    lda_model = models.ldamodel.LdaModel(
        corpus=corpus, id2word=dictionary, num_topics=groups,
        passes=10, iterations=200, alpha="auto", eta="auto"
    )

    group_results = []
    for sents in reviews:
        single_corpus = [dictionary.doc2bow(sent) for sent in sents]
        sents_group_result = []
        for scores in lda_model.inference(single_corpus)[0]:
            max_score = max(scores)
            max_index = scores.argmax()

            # 設定閾值，過低的分數視為無意義
            if max_score < 0.2:  # 閾值可調整
                sents_group_result.append(0)
            else:
                sents_group_result.append(max_index + 1)

        group_results.append(sents_group_result)
    
    return group_results, lda_model


def pad_and_trunc(group_results, *, max_sentence):
    #max number of sentences in a review
    result_list = []
    for i, result in enumerate(group_results):
        if len(result) >= max_sentence:
            result = result[:max_sentence]
        else:
            result.extend([0]*(max_sentence-len(result)))
        result_list.append(np.array(result).astype(int))
    return result_list

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
# You can repeatly try until satisfied 
# 5mins
GROUPS = 5
clean_review_texts = lemmatizer_with_delete_stopword(df_review["split_review_text"].tolist())
group_list, lda_model = LDAGrouping(clean_review_texts, GROUPS) 
pad_group_list = pad_and_trunc(group_list, max_sentence=args["max_sentence"])
df_review["LDA_group"] = pad_group_list
df_review

,user_id,asin,rating,like,review_text,split_review_text,LDA_group
2423565,AE22UDPZNRFQPRKMKKP5QMRB3TTA,B07H9XB7G6,3.0,False,This works well but it’s very bulky and heavy....,[This works well but it’s very bulky and heavy...,"[2, 2, 5, 0, 0, 0, 0, 0, 0, 0]"
2423569,AE22UDPZNRFQPRKMKKP5QMRB3TTA,B073CWSQ51,5.0,True,Works great. Works great in this Florida humid...,"[Works great., Works great in this Florida hum...","[2, 2, 3, 2, 3, 1, 0, 0, 0, 0]"
2423566,AE22UDPZNRFQPRKMKKP5QMRB3TTA,B000WI02HG,5.0,True,This is the best brand for getting those hairs...,[This is the best brand for getting those hair...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2423563,AE22UDPZNRFQPRKMKKP5QMRB3TTA,B07X6HCLK2,5.0,True,"So smooth I love it. I shampooed, lathered thi...","[So smooth I love it., I shampooed, lathered t...","[2, 5, 3, 3, 0, 0, 0, 0, 0, 0]"
2423575,AE22UDPZNRFQPRKMKKP5QMRB3TTA,B00390DN34,5.0,True,Big hair doesn't add weight,[Big hair doesn't add weight],"[2, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
...,...,...,...,...,...,...,...
5105234,AHZZM2KGP52A6KBAS2IZ2RVZUJJA,B000YJ2SLG,1.0,False,This is a bad product. It floats on the skin a...,"[This is a bad product., It floats on the skin...","[2, 2, 2, 2, 0, 0, 0, 0, 0, 0]"
5105231,AHZZM2KGP52A6KBAS2IZ2RVZUJJA,B0062IWQVU,3.0,False,"Works OK. Nothing special. Small, trims uneven...","[Works OK., Nothing special., Small, trims une...","[2, 2, 1, 2, 2, 4, 0, 0, 0, 0]"
5105219,AHZZM2KGP52A6KBAS2IZ2RVZUJJA,B008QMWM5U,2.0,False,Bar gets slimy quickly and take a lot of water...,[Bar gets slimy quickly and take a lot of wate...,"[2, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
6832816,AHZZNS4SMZYTNG2DPGJRXJDRPRWQ,B01499YQP2,5.0,True,Hard hitters.,[Hard hitters.],"[2, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


##### Print topics trained from LDA


In [17]:
print('Topic: 0 \ndefault padding group')
for idx, topic in lda_model.show_topics(formatted=False, num_words= 30):
    print('Topic: {} \nWords: {}'.format(idx+1, '|'.join([w[0] for w in topic])))

Topic: 0 
default padding group
Topic: 1 
Words: good|just|nail|dont|brush|price|buy|better|need|bought|bottle|try|brand|lot|got|come|think|quality|review|pretty|set|thing|sure|want|right|far|job|know|money|size
Topic: 2 
Words: product|great|skin|like|love|work|really|smell|doesnt|feel|little|long|oil|scent|recommend|bit|way|stuff|definitely|light|amazing|say|sensitive|lotion|happy|purchase|highly|leave|oily|body
Topic: 3 
Words: hair|time|makeup|best|tried|lash|mascara|perfect|apply|polish|gel|fine|natural|hold|coat|hard|end|different|easily|water|remove|shampoo|curl|conditioner|second|minute|glue|needed|add|heat
Topic: 4 
Words: dry|nice|easy|soft|hand|clean|super|smooth|favorite|fast|shower|soap|foot|store|shave|wet|razor|thank|bristle|quick|case|shiny|close|gentle|nicely|extremely|shaving|hope|cause|reason
Topic: 5 
Words: use|using|used|make|face|ive|day|color|look|eye|year|cream|help|week|looking|wash|powder|stay|spray|moisturizer|foundation|lip|month|result|night|difference|wea

##### 取得每個topic的 word_list

In [18]:
# 取每個 topic 的前 30 個詞 & 機率
topics = [lda_model.show_topics(formatted=False, num_words=30)[i][1] for i in range(len(lda_model.show_topics()))]

words_lists = [[word for word, _ in topic] for topic in topics]

Z1, Z2, Z3, Z4, Z5 = words_lists

##### flan-t5

In [97]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-large"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

keywords = Z1
input_text = f"Summarize the following keywords into a meaningful sentence: {', '.join(keywords)}"

input_ids = tokenizer(input_text, return_tensors="pt").input_ids
output = model.generate(input_ids, max_length=100, num_return_sequences=1)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_text)

i just really like this mascara because it has great quality and a great scent. it's a little bit more expensive than other mascaras but it's worth it because it lasts longer and it doesn't smudge.


In [98]:
keywords = Z2
input_text = f"Summarize the following keywords into a meaningful sentence: {', '.join(keywords)}"

input_ids = tokenizer(input_text, return_tensors="pt").input_ids
output = model.generate(input_ids, max_length=100, num_return_sequences=1)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_text)

make your own makeup with a bit of cream and a little bit of powder to make your face look nice and smooth without looking like you have dry skin. this is a great way to make your own makeup at home and is very easy to apply and doesnt require a lot of skill.


In [99]:
keywords = Z3
input_text = f"Summarize the following keywords into a meaningful sentence: {', '.join(keywords)}"

input_ids = tokenizer(input_text, return_tensors="pt").input_ids
output = model.generate(input_ids, max_length=100, num_return_sequences=1)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_text)

A man is giving a woman a blow dryer with a little bit of heat and a little bit of spf in his hair. He takes a few seconds to let the hair dry, then gives her a little bit of heat and a little bit of spf in his hair. He then takes a few seconds to let the hair dry, then gives her a little bit of heat and a little bit of spf


In [ ]:
keywords = Z4
input_text = f"Summarize the following keywords into a meaningful sentence: {', '.join(keywords)}"
# print(input_text)
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
output = model.generate(input_ids, max_length=100, num_return_sequences=1)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_text)

Summarize the following keywords into a meaningful sentence: nail, color, polish, gel, natural, moisturizing, soap, dark, beautiful, shave, razor, awesome, shade, brown, theyre, file, muy, shaving, home, okay, reason, tan, decent, impressed, fan, medium, deep, bath, complaint, pink
a man with a pink tan and a dark brown beard is shaving his beard with a razor and a file.


In [101]:
keywords = Z5
input_text = f"Summarize the following keywords into a meaningful sentence: {', '.join(keywords)}"

input_ids = tokenizer(input_text, return_tensors="pt").input_ids
output = model.generate(input_ids, max_length=100, num_return_sequences=1)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_text)

ive been using this product for a week and i love it ive been using it for a month and i have been trying to find a bottle of the same oil ive been using for a year and i have never had any issues with it .
